In [ ]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;

In [1]:
from IPython.core.interactiveshell import InteractiveShell  
InteractiveShell.ast_node_interactivity = "all" 
from IPython.display import display, HTML 
display(HTML("<style>.container { width:90% !important; }</style>")) 

In [ ]:
from dash import Dash, html, dcc, Input, Output, dash_table, callback, State, ClientsideFunction
from dash.exceptions import PreventUpdate
import dash_bootstrap_components as dbc
import collections
import pandas as pd
import wget
import json
import random
from application.dash.biocodex.functions import join_id_adr_cdb, build_one
from application.dash.sidebar import spe_options, uga_options, ciblage_options 
from dash_extensions.javascript import assign, arrow_function, Namespace
import dash_leaflet as dl
import dash_leaflet.express as dlx
import re
from datetime import datetime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import Column, Integer, Sequence, String, Text, DateTime, ForeignKey, create_engine
import datetime
from flask import Flask, request, url_for
from datatables import DataTable
from application.dash.biocodex.functions import mean_lat, mean_lon, datatable_cols, styles, legend1, legend2, data_df, prepare_data, display_cols

In [ ]:
from application.config import basedir
from dotenv import load_dotenv
import os
load_dotenv(os.path.join(basedir, '.env'))
DATABASE_URL = os.environ.get('DATABASE_URL').replace('postgres://', 'postgresql://')
engine = create_engine(DATABASE_URL)

In [2]:
from application.dash.biocodex.functions import join_id_adr_cdb, build_one, prepare_data, display_cols

In [3]:
df = join_id_adr_cdb()

In [4]:
data_df = prepare_data(df)

In [5]:
data_df = data_df[display_cols].copy()

In [6]:
data_df.columns = ['id', 'nom', 'prenom', 'spe', 'pot', 'pvm', 'nv2', 'cib', 'etablissement', 'uga', 'adresse', 'cp', 'ville', 'tel', 'mode', 'com', 'ddv', 'dpv', 'rdv']

In [7]:
for col in ['ddv', 'dpv']:
    data_df[col] = np.where(data_df[col]==0, '', data_df[col])

In [8]:
datatable_cols =[{"name": i.upper(), "id": i} for i in data_df.columns]

In [29]:
data_df["rdv"] = [dbc.Checkbox(id=f"checkbox-{i}", label="", value=True) if row['dpv'] != '' else dbc.Checkbox(id=f"checkbox-{i}", label="", value=False) for i, row in data_df.iterrows()]

In [36]:
data_df.loc[5, 'rdv']

Checkbox(id='checkbox-5', label='', value=True)

In [10]:
data = data_df.to_dict('records')

In [11]:
len(data)

2078

In [ ]:
from app import app
app.app_context().push()

In [17]:
import dash_bootstrap_components as dbc

In [20]:
from dash import html

In [23]:
from application.dash.biocodex.functions import doctor_colors

In [24]:
def build_one(row):

    return dbc.Col(
    [
        dbc.Card(
            [
                html.Div(
                    [
                        row["spe"]
                    ],
                    className="badge position-absolute top-0 start-0 translate-middle",
                    style={
                        "color": "#FFFFFF",
                        "fontWeight": 900,
                        "fontSize": "0.75rem",
                        "background-color": f"{doctor_colors[row['spe']]}",
                    },
                    **{"data-tor": "place.left place.top"}
                ),
                html.Span([f"{row['cib']}", html.Span([], className="visually-hidden")],
                          className="position-absolute top-0 start-100 translate-middle badge rounded-pill bg-dark",
                          style={"font-size": "14px"}),
                dbc.CardHeader(
                    [
                        html.Div(
                            [
                                html.Img(
                                    width=32,
                                    height=32,
                                    src=f"/home/julien/website/application/static/img/{row['spe']}.jpg",
                                    className="avatar-md rounded-circle img-thumbnail"
                                ),
                                html.A(
                                    [
                                    ],
                                    className="text-dark",
                                    href=f"/biocodex/{row['id']}",

                                )
                            ], className="d-flex align-items-center"
                        )
                    ]
                ),
                dbc.CardBody(
                    [
                        dbc.Col([row["mode"]], width=6, className="border border-secondary-subtle d-flex justify-content-center"),
                        dbc.Col([row["com"]], width=12, className="border border-secondary-subtle d-flex justify-content-center", style={"line-height": "18px"}),
                        dbc.Col([row["ddv"]], width=6, className="border border-secondary-subtle d-flex justify-content-center"),
                        dbc.Col([row["dpv"]], width=6, className="border border-secondary-subtle d-flex justify-content-center"),
                    ],
                    className="d-flex flex-wrap"
                ),
            ]
        )
    ],
        className="d-inline-block m-3 col-xl-2 col-sm-6",
)

In [25]:
build_one(data[0])

Col(children=[Card([Div(children=['MG'], className='badge position-absolute top-0 start-0 translate-middle', style={'color': '#FFFFFF', 'fontWeight': 900, 'fontSize': '0.75rem', 'background-color': '#aaa'}, data-tor='place.left place.top'), Span(children=['4', Span(children=[], className='visually-hidden')], className='position-absolute top-0 start-100 translate-middle badge rounded-pill bg-dark', style={'font-size': '14px'}), CardHeader([Div(children=[Img(className='avatar-md rounded-circle img-thumbnail', height=32, src='/home/julien/website/application/static/img/MG.jpg', width=32), A(children=[], className='text-dark', href='/biocodex/1')], className='d-flex align-items-center')]), CardBody(children=[Col(children=['L'], className='border border-secondary-subtle d-flex justify-content-center', width=6), Col(children=[''], className='border border-secondary-subtle d-flex justify-content-center', style={'line-height': '18px'}, width=12), Col(children=['21/07/2023 14:00'], className='b

In [ ]:
import visdcc
import dash

In [ ]:
def generate_html_table_from_df(df, id):
    Thead = html.Thead(
        [html.Tr([html.Th(col) for col in df.columns])]
    )
    Tbody = html.Tbody(
        [html.Tr(
            [html.Td( df.iloc[i, j], id = '{}_{}_{}'.format(id, i, j) ) for j in range(len(df.columns))]
         ) for i in range(len(df))]
    )
    return html.Table([Thead, Tbody], id = id, className = "display")

df = pd.DataFrame({'name': ['Jacky', 'Mei', 'Jay', 'Sandy', 'Jerry', 'Jimmy', 'Jeff',
                            'Jacky', 'Mei', 'Jay', 'Sandy', 'Jerry', 'Jimmy', 'Jeff',
                            'Jacky', 'Mei', 'Jay', 'Sandy', 'Jerry', 'Jimmy', 'Jeff'],
                   'age': [18, 71, 14, 56, 22, 28, 15,
                           18, 71, 14, 56, 22, 28, 15,
                           18, 71, 14, 56, 22, 28, 15]}, columns = ['name', 'age'])

external_scripts = ['https://code.jquery.com/jquery-3.3.1.min.js',
                    'https://cdn.datatables.net/v/dt/dt-1.10.18/datatables.min.js']
external_stylesheets = ['https://cdn.datatables.net/v/dt/dt-1.10.18/datatables.min.css']

app = Dash(__name__, 
    external_scripts = external_scripts,
    external_stylesheets = external_stylesheets
)

app.layout = html.Div([
    html.Button('apply datatable', id = 'button'),
    visdcc.Run_js(id = 'javascript'),
    html.Br(),
    html.Div(
        generate_html_table_from_df(df, id = 'datatable'), 
        style = {'width': '40%'}
    ),
    html.H1(id = 'output_div')
])
           
@app.callback(
    Output('javascript', 'run'),
    [Input('button', 'n_clicks')])
def myfun(x): 
    if x: 
        return "$('#datatable').DataTable()"
    return ""

@app.callback(
    Output('output_div', 'children'),
    [Input('datatable_{}_{}'.format(i, j), 'n_clicks') for i in range(len(df)) for j in range(len(df.columns))])
def myfun(*args): 
    ctx = dash.callback_context
    if not ctx.triggered or ctx.triggered[0]['value'] is None:  return ""
    input_id = ctx.triggered[0]['prop_id'].split('.')[0]
    row = input_id.split('_')[1]
    col = input_id.split('_')[2]
    return "you click on row {} col {}".format(row, col)

In [ ]:
app.run_server(debug=True)

In [ ]:
app.get_asset_url('img/PE.jpg')

In [ ]:
$(document).ready(function() {
    $('#datatable').DataTable( {
        "bProcessing": true,
        "bServerSide": true,
        "sAjaxSource": "application/data/databank_indicators.json"
    } );
} );

In [ ]:
button = dbc.Button([html.Span([html.Span([], className="visually-hidden")], className="position-absolute top-0 start-100 translate-middle p-2 bg-danger border border-light rounded-circle")], type="buttonj", className="btn btn-primary position-relative")

In [ ]:
card = dbc.Col(
    [
        dbc.Card(
            [
                dbc.CardBody(
                    [
                        html.Div(
                            [
                                html.Img(
                                    width=32,
                                    height=32,
                                    src=f"./application/static/img/{row['spe']}.jpg",
                                    className="avatar-md rounded-circle img-thumbnail"
                                ),
                                html.A(
                                    [
                                        html.H6([row["nom"], html.Span(' '), row["prenom"].title()], className="mb-0")
                                    ],
                                    className="text-dark",
                                    href=f"/dash/biocodex/table/{row['id']}",
                                    style={"fontSize": "0.7rem"}
                                )
                            ], className="d-flex align-items-center"
                        ),
                        html.Div(
                            [

                                html.Span( row["spe"],className="badge badge-success mb-0")
                            ],
                            className="flex-1 ms-3"
                        ),
                        html.Div([]),
                    ]
                )
            ]
        )
    ],
        className="d-inline-block m-3 col-xl-2 col-sm-6",
)

In [ ]:
def json_js_to_geojson(json_js_file):
    import json

    with open(f'assets/js/{json_js_file}.js') as dataFile:
        data = dataFile.read()

        obj = data[data.find('{') : data.rfind('}')+1]

        geojson = json.loads(obj)

    with open(f'assets/{json_js_file}.json', 'w') as j:
            j.write(json.dumps(geojson).replace(": ", ":").replace(", ", ","))
            
    return geojson

def unix_to_dt(unx_str):
    return datetime.utcfromtimestamp(int(unx_str)).strftime('%d/%m/%Y %H:%M')

def p_popup(pharmacy):
    return f"""
            <b>{pharmacy['nom']}</b>
            <br>{pharmacy["adresse"]}
            <br>{pharmacy["cp"]} {pharmacy["ville"]} ({pharmacy["uga"]})
            <br>{pharmacy["tel"]}
            <br>CA UL (CMA juin 23): {pharmacy["ca ul cma juin 23"]} (rang {pharmacy["rang ca uk"]})
            <br>Ciblage DP:{pharmacy["ciblage dp"]} DSO:{pharmacy["ciblage dso"]} DM:{pharmacy["ciblage dm"]}                
        """

def c_popup(doc):
    reg=re.compile('[0-9]{10}')
    if isinstance(doc['mode'], str) and re.search(reg, doc['mode']):
        doc['mode']=unix_to_dt(doc['mode'])

    if doc['ddv']:
        doc['ddv']=unix_to_dt(doc['ddv'])

    return f"""
        <span class="me-1 badge bg-info">{doc['doc_id']}</span>
        <b><h5>{doc['nom']} </b> {doc['prenom']}</h5>
        <br><b>{doc["spe"]} </b>
        <br>Potentiel: {doc["pot"]} PVM: {doc["pvm"]}</p>
        <p>{doc["etablissement"]}
        <br>{doc["adresse"]}
        <br>{doc["cp"]} {doc["ville"]} ({doc["uga"]})
        <br><i class="fa fa-duotone fa-phone"></i> {doc["tel"]}</p>
        <hr>
        <p>Ciblage: {doc["ciblage"]}
        <br>DDV: {doc["ddv"]}
        <br>NV2022: {doc["nv2022"]}</p>
        <p>Mode: {doc["mode"]}
        <br>Commentaire: {doc["commentaire"]}</p>
        """

def get_compo(df):    
    idx = pd.IndexSlice
    compo = df.pivot_table(values="nom", index=["uga", "spe"], columns="ciblage", aggfunc='count').fillna(0).astype(int)
    compo = compo[['2', '3', '4', '']]
    compo.columns=["2x", "3x", "4x", "non ciblé"]
    compo.index.name = None
    new_index = pd.MultiIndex.from_product([
        ugas,
        spes
    ], names=["uga", "spe"])
    compo.reindex(new_index)
    
    return compo

def get_uga_compo(compo, uga):
    compo = compo.loc[[uga,],:].reset_index().drop("uga", axis=1)
    for col in ["2x", "3x", "4x"]:
        compo[col] = np.where(compo[col]==0, '', compo[col])
        
    return compo

def get_info(feature=None):
    header = [html.H4("Composition")]
    if not feature:
        return header + [html.P("Hoover over a UGA")]
    uga_compo = get_uga_compo(get_compo(df), feature["properties"]["CODE_UGA"])
    table = dash_table.DataTable(
        uga_compo.to_dict('records'),
        [{"name": i.upper(), "id": i} for i in uga_compo.columns],
        style_cell_conditional=[
            {
                'if': {'column_id': 'spe'},
                'textAlign': 'left'
            }
        ],
        style_header={ 'width': '50px', 'border': '1px solid black' , 'textAlign': 'center'},
        style_cell={ 'width': '50px', 'border': '1px solid grey', 'textAlign': 'center' },
    )
    return header + [html.B(feature["properties"]["CODE_UGA"]), table]


In [ ]:
ugas = ["75AUT", "75PAS", "75TRO", "75INV", "75ELY", "75GRE", "75VAU", "75MNP", "75PER", "75TER", "92LEV", "92NEU"]
spes = ["GY", "MG-GY", "SF", "MG", "GE", "PE", "PE-PSY", "PSY", "NE"]
df = join_id_adr_cdb()

    for col in ['nv2022', 'ciblage']:
        df[col] = np.where(df[col]==0, '', df[col])

    df_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df.to_dict('records')])

    with open(f'assets/df_geojson.json', 'w') as j:
        j.write(json.dumps(df_geojson).replace(": ", ":").replace(", ", ","))

    target_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df[df['ciblage']!=''].to_dict('records')])

    with open(f'assets/target_geojson.json', 'w') as j:
        j.write(json.dumps(target_geojson).replace(": ", ":").replace(", ", ","))

    with open(f'app/dashboards/assets/target_geojson.json', 'w') as j:
        j.write(json.dumps(target_geojson).replace(": ", ":").replace(", ", ","))

    untarget_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df[df['ciblage']==''].to_dict('records')])

    with open(f'assets/untarget_geojson.json', 'w') as j:
        j.write(json.dumps(untarget_geojson).replace(": ", ":").replace(", ", ","))

    with open(f'app/dashboards/assets/untarget_geojson.json', 'w') as j:
        j.write(json.dumps(untarget_geojson).replace(": ", ":").replace(", ", ","))

In [ ]:
from application.dash.biocodex.functions import join_id_adr_cdb
df = join_id_adr_cdb()

In [ ]:
df.sample()

In [ ]:
mean_lat = df['lat'].mean()
mean_lon = df['lon'].mean()

In [ ]:
with open("assets/uga_gpd.json", "r") as uga_json:
    uga_geojson = json.loads(uga_json.read())

sector_features = [uga_geojson["features"][i] for i in range(len(uga_geojson["features"])) if uga_geojson["features"][i]["properties"]["CODE_UGA"] in ugas]
uga_geojson["features"] = sector_features

with open("assets/pharmas_geojson.json", "r") as pharma_json:
    pharma_geojson = json.loads(pharma_json.read())
    
with open("assets/target_geojson.json", "r") as cible_json:
    target_geojson = json.loads(cible_json.read())
    
with open("assets/untarget_geojson.json", "r") as cible_json:
    untarget_geojson = json.loads(cible_json.read())

In [ ]:
ns = Namespace('dashExtensions','default')

ugas_layer = dl.GeoJSON(
    data=uga_geojson,
    hideout=dict(selected=[]),
    filter=ns('uga_geojson_filter'),
    style=ns('uga_style_handle'),
    hoverStyle=arrow_function(dict(weight=3, color='red', dashArray='')),
    zoomToBounds=True,
    id="uga-geojson",
)

pharmas_layer = dl.GeoJSON(
    data=pharma_geojson,
    hideout=dict(ugas_selected=[]),
    filter=ns('pharma_filter'),
    zoomToBounds=True,
    pointToLayer=ns('pharma_icon'),
    id="pharma-geojson"
)

target_layer = dl.GeoJSON(
    data=target_geojson,
    hideout=dict(ugas_selected=[], spes_selected=[]),
    filter=ns('geojson_filter'),
    zoomToBounds=True,
    pointToLayer=ns('cible_icon'),
    id="target-geojson"
)

untarget_layer = dl.GeoJSON(
    data=untarget_geojson,
    hideout=dict(ugas_selected=[], spes_selected=[]),
    filter=ns('geojson_filter'),
    zoomToBounds=True,
    pointToLayer=ns('cible_icon'),
    id="untarget-geojson"
)

info = html.Div(
    children=get_info(), 
    id="info", 
    className="info",
    style={"position": "absolute", "bottom": "10px", "left": "10px", "zIndex": "1000"}
)

In [ ]:
external_scripts=["js/leaflet.extra-markers.min.js"]
external_stylesheets=[dbc.themes.BOOTSTRAP, dbc.icons.FONT_AWESOME, "css/leaflet.extra-markers.min.css"]

app = Dash(external_scripts=external_scripts, external_stylesheets=external_stylesheets)

app.layout = html.Div(
    [
        dcc.Checklist(id="uga-cl", value=["75AUT"], options=[{"value": uga, "label": uga} for uga in ugas], inline=True),
        dcc.Checklist(id="spe-cl", value=["GY"], options=[{"value": spe, "label": spe} for spe in df["spe"].unique()], inline=True),
        dl.Map(
            [
               dl.LayersControl(
                    [
                        dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                        dl.Overlay(dl.LayerGroup(ugas_layer, id="ugas_layer_group"), name="ugas", checked=True),
                        dl.Overlay(dl.LayerGroup(pharmas_layer, id="pharmas_layer_group"), name="pharmas", checked=True),
                        dl.Overlay(dl.LayerGroup(target_layer, id="target_layer_group"), name="ciblés", checked=True),
                        dl.Overlay(dl.LayerGroup(untarget_layer, id="untarget_layer_group"), name="non ciblés", checked=False),
                        info
                    ]
               ),
                dl.FullScreenControl(),
                dl.LocateControl(locateOptions={'enableHighAccuracy': True})               
           ],
           center=(mean_lat, mean_lon), 
           zoom=12, 
           style={'height': '90vh'}
        )
    ]
)

@app.callback(
    Output("uga-geojson", "hideout"),
    Input("uga-cl", "value"),
    State("uga-geojson", "hideout"),
)
def toggle_select(ugas_selected, uga_hideout):
    uga_hideout["selected"] = ugas_selected
    return uga_hideout

@app.callback(
    Output("pharma-geojson", "hideout"),
    Input("uga-cl", "value"),
    State("pharma-geojson", "hideout")
)
def toggle_select(ugas_selected, pharma_hideout):
    pharma_hideout["ugas_selected"] = ugas_selected
    return pharma_hideout

@app.callback(
    Output("target-geojson", "hideout"),
    Input("uga-cl", "value"),
    Input("spe-cl", "value"),
    State("target-geojson", "hideout")
)
def toggle_select(ugas_selected, spes_selected, target_hideout):
    target_hideout["ugas_selected"] = ugas_selected
    target_hideout["spes_selected"] = spes_selected
    return target_hideout

@app.callback(
    Output("untarget-geojson", "hideout"),
    Input("uga-cl", "value"),
    Input("spe-cl", "value"),
    State("untarget-geojson", "hideout")
)
def toggle_select(ugas_selected, spes_selected, untarget_hideout):
    untarget_hideout["ugas_selected"] = ugas_selected
    untarget_hideout["spes_selected"] = spes_selected
    return untarget_hideout

@app.callback(
    Output("info", "children"), 
    Input("uga-geojson", "hoverData")
)
def info_hover(feature):
    return get_info(feature)

    
app.run_server(debug=True)

In [ ]:
app = Dash(name="maps_app", , "css/geojson.css"])

app.layout = html.Div(
    [
        dcc.Checklist(id="uga-cl", value=uga_cl_default, options=uga_cl_opt, inline=True, labelStyle={"display": "inline-block"}),        
        
        dbc.Button("Show UGAs", id="uga-btn"),
        dbc.Button("Show pharmacies", id="pharma-btn"),
        dbc.Button("Show doctors", id="doc-btn"),
        dl.Map(
            [
                dl.LayersControl(
                    [
                        dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                        dl.Overlay(
                            dl.LayerGroup(
                                ugas_layer,
                                id="ugas_layer_group"
                            ),
                            name="ugas",
                            checked=True
                        ),
                        dl.Overlay(
                            dl.LayerGroup(
                                pharmas_layer,
                                id="pharmas_layer_group"
                            ),
                            name="pharmacies",
                            checked=True
                        ),
                        dl.Overlay(
                            dl.LayerGroup(
                                docs_layer,
                                id="docs_layer_group"
                            ),
                            name="doctors",
                            checked=True
                        )
                    ]
                ),
                dl.FullScreenControl(),
                dl.LocateControl(locateOptions={'enableHighAccuracy': True}),
                info
            ],
            center=(ciblage["lat"].mean(), ciblage["lon"].mean()), 
            zoom=11, 
            style={'height': '90vh'}
        )
    ]
)




app.clientside_callback(
    "function(x){return x;}", 
    Output("uga_geojson", "hideout"), 
    Input("uga-cl", "value")
)





    @app.callback(Output("ugas_layer_group", "children"), Input("uga-btn", "n_clicks"))
    def gen_uga_layer(_):
        return ugas_layer

    @app.callback(Output("pharmas_layer_group", "children"), Input("pharma-btn", "n_clicks"))
    def gen_uga_layer(_):
        return pharmas_layer

    @app.callback(Output("docs_layer_group", "children"), Input("doc-btn", "n_clicks"))
    def gen_uga_layer(_):
        return docs_layer
    
    app.clientside_callback(
    "function(x){return x;}", 
    Output("doc_geojson", "hideout"), 
    Input("uga-cl", "value")
    )
    
    app.clientside_callback(
        "function(x){return x;}", 
        Output("pharma_geojson", "hideout"), 
        Input("uga-cl", "value")
    )

In [ ]:
app.run_server(debug=True,dev_tools_hot_reload=True)

m = Map(center=(ciblage["lat"].mean(), ciblage["lon"].mean()), zoom=12)


for uga in ugas:
    layers=[]
    uga_data = {
        'type': data['type'],
        'crs': data['crs'],
        'features': [data['features'][i] for i in range(len(data['features'])) if data['features'][i]['properties']['CODE_UGA'] == uga]
    }
    geo_json = GeoJSON(
        data=uga_data,
        style={
            'opacity': 1, 'dashArray': '9', 'fillOpacity': 0.1, 'weight': 1
        },
        hover_style={
            'color': 'white', 'dashArray': '0', 'fillOpacity': 0.5
        },
        style_callback=random_color,
        zoomToBounds=True
    )
    layers.append(geo_json)
    pharma_markers=[]
    for i, pharmacy in pharma[pharma["uga"]==uga].iterrows():

        popup = HTML()
        popup.value = f"""
            <b>{pharmacy['nom']}</b>
            <br>{pharmacy["adresse"]}
            <br>{pharmacy["cp"]} {pharmacy["ville"]} ({pharmacy["uga"]})
            <br>{pharmacy["tel"]}
            <br>CA UL (CMA juin 23): {pharmacy["ca ul cma juin 23"]} (rang {pharmacy["rang ca uk"]})
            <br>Ciblage DP:{pharmacy["ciblage dp"]} DSO:{pharmacy["ciblage dso"]} DM:{pharmacy["ciblage dm"]}                
        """

        marker = Marker(
            icon = AwesomeIcon(
                name='plus-square',
                marker_color='green',
                icon_color='white',
                spin=False
            ), 
            location=(pharmacy["lat"], pharmacy["lon"]),
            popup=popup

        )    

        pharma_markers.append(marker)
        pharma_cluster=MarkerCluster(markers=(pharma_markers), name="pharma")
    layers.append(pharma_cluster)
        
    for spe in spe_to_color.keys():
        cible_spe=ciblage[(ciblage["spe"]==spe)&(ciblage["uga"]==uga)].copy()
        spe_markers=[]
        for i, doc in cible_spe.iterrows():
            marker = Marker(
                icon=AwesomeIcon(
                    name='user-md',
                    marker_color=spe_to_color[doc["spe"]]
                ), 
                location=(doc["lat"], doc["lon"])
            )
            spe_markers.append(marker)
        spe_cluster=MarkerCluster(markers=(spe_markers), name=spe)
        layers.append(spe_cluster)
    
    layer_group=LayerGroup(layers=(layers), name=uga)

    m.add_layer(layer_group)


m.add_control(LayersControl(position='topright'))
m.add_control(FullScreenControl())
m.add_control(ScaleControl(position='bottomleft'))

marker = Marker(icon=AwesomeIcon(name="check", marker_color='green', icon_color='darkred'))

m.add_control(SearchControl(
  position="topleft",
  layer=layer_group,
  zoom=15,
  marker=marker
))

display(m)

    import random

    number_of_colors = len(ugas)
    colors=[]

    color = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)]) for i in range(number_of_colors)]
    colors.append(color)
    colors=colors[0]

    uga_colors = {}
    for u, c in zip(ugas, colors):
        uga_colors[u]=c